# Question 1

In [1]:
# Import Spark Session
from pyspark.sql import SparkSession

# Create the Spark Session
spark = SparkSession.builder.appName("CA2_SeoulBike").getOrCreate()            

spark.sparkContext.setLogLevel("ERROR")                               # Only show errors messages

bike_df = spark.read.csv("hdfs:///CA2_SBA25076/SeoulBikeData.csv",    # Read the dataset on HDFS
                    header=True,                                      # Keep the header 
                    inferSchema=True)                                 # Infer the type of each column

bike_df.show(5)                                                       # Show the first 5 rows
bike_df.printSchema()                                                 # Print the type of each column

26/07/02 21:27:51 WARN Utils: Your hostname, DESKTOP-DUGUN4Q resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/02 21:27:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 21:28:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------+-----------------+----+---------------+-----------+----------------+----------------+-------------------------+-----------------------+------------+-------------+-------+----------+---------------+
|      Date|Rented Bike Count|Hour|Temperature(�C)|Humidity(%)|Wind speed (m/s)|Visibility (10m)|Dew point temperature(�C)|Solar Radiation (MJ/m2)|Rainfall(mm)|Snowfall (cm)|Seasons|   Holiday|Functioning Day|
+----------+-----------------+----+---------------+-----------+----------------+----------------+-------------------------+-----------------------+------------+-------------+-------+----------+---------------+
|01/12/2017|              254|   0|           -5.2|         37|             2.2|            2000|                    -17.6|                    0.0|         0.0|          0.0| Winter|No Holiday|            Yes|
|01/12/2017|              204|   1|           -5.5|         38|             0.8|            2000|                    -17.6|                    0.0|         0.0|

In [2]:
# Rename corrupted names of columns or with space (without measure unit)
bike_df = bike_df.withColumnRenamed("Temperature(�C)", "Temperature_C") \
       .withColumnRenamed("Dew point temperature(�C)", "DewPoint_C") \
       .withColumnRenamed("Rented Bike Count", "Rented_Bike_Count") \
       .withColumnRenamed("Functioning Day", "Functioning_Day")

In [3]:
# Check the changes
bike_df.printSchema()

# Although Spark automatically infers data types when reading the dataset, 
# it is necessary to verify whether the inferred types are accurate.

root
 |-- Date: string (nullable = true)
 |-- Rented_Bike_Count: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Temperature_C: double (nullable = true)
 |-- Humidity(%): integer (nullable = true)
 |-- Wind speed (m/s): double (nullable = true)
 |-- Visibility (10m): integer (nullable = true)
 |-- DewPoint_C: double (nullable = true)
 |-- Solar Radiation (MJ/m2): double (nullable = true)
 |-- Rainfall(mm): double (nullable = true)
 |-- Snowfall (cm): double (nullable = true)
 |-- Seasons: string (nullable = true)
 |-- Holiday: string (nullable = true)
 |-- Functioning_Day: string (nullable = true)



####
After reviewing the inferred schema, only minor corrections were required. Most columns were already assigned appropriate numeric or string data types, while the main adjustments involved formatting the date field.

In [4]:
# Adjust date field
import pyspark.sql.functions as sf

bike_df = bike_df.withColumn("Date", sf.to_date("Date", "dd/MM/yyyy"))

In [5]:
# Check the change

bike_df.select("Date").printSchema()

root
 |-- Date: date (nullable = true)



In [6]:
# Before check duplicates

print(f"The dataset has {bike_df.count()} rows")

The dataset has 8760 rows


In [7]:
# Drop Duplicates

bike_df = bike_df.dropDuplicates()

In [8]:
# After check duplicates

print(f"The dataset has {bike_df.count()} rows")

The dataset has 8760 rows


In [9]:
# Check null values
from pyspark.sql.functions import when, count, col


bike_df.select([count(when(col(c).isNull(), c)).alias(c) for c in bike_df.columns]).show()

+----+-----------------+----+-------------+-----------+----------------+----------------+----------+-----------------------+------------+-------------+-------+-------+---------------+
|Date|Rented_Bike_Count|Hour|Temperature_C|Humidity(%)|Wind speed (m/s)|Visibility (10m)|DewPoint_C|Solar Radiation (MJ/m2)|Rainfall(mm)|Snowfall (cm)|Seasons|Holiday|Functioning_Day|
+----+-----------------+----+-------------+-----------+----------------+----------------+----------+-----------------------+------------+-------------+-------+-------+---------------+
|   0|                0|   0|            0|          0|               0|               0|         0|                      0|           0|            0|      0|      0|              0|
+----+-----------------+----+-------------+-----------+----------------+----------------+----------+-----------------------+------------+-------------+-------+-------+---------------+



#### After checking for null and duplicate values, none were found, so there is no need to drop them.

In [10]:
# Check unique values

bike_df.select("Seasons").distinct().show()
bike_df.select("Holiday").distinct().show()
bike_df.select("Functioning_Day").distinct().show()

+-------+
|Seasons|
+-------+
| Spring|
| Summer|
| Autumn|
| Winter|
+-------+

+----------+
|   Holiday|
+----------+
|No Holiday|
|   Holiday|
+----------+

+---------------+
|Functioning_Day|
+---------------+
|             No|
|            Yes|
+---------------+



### Explore central tendency and dispersion (mean, std, percentiles) of the key numeric features

In [11]:
from pyspark.sql import functions as sf

# Descriptive statistics for bike rental variable
rental_stats = bike_df.select("Rented_Bike_Count").summary()
rental_stats.show()

+-------+-----------------+
|summary|Rented_Bike_Count|
+-------+-----------------+
|  count|             8760|
|   mean|704.6020547945205|
| stddev|644.9974677392166|
|    min|                0|
|    25%|              191|
|    50%|              504|
|    75%|             1064|
|    max|             3556|
+-------+-----------------+



#### Descriptive statistics for hourly bike rentals

From the 8,760 records obtained, it was identified that, on average, **about 705 bikes are rented per hour**. However, there is already **very high variability in these values, with a standard deviation of approximately 645 rentals**, indicating that in some hours **no bikes are rented at all**. This suspicion is confirmed by the **minimum value equal to 0**.

By analysing the quartiles, this high variability is reinforced: the **difference between the third quartile (1,064 rentals) and the first quartile (191 rentals) is 873 rentals**. In addition, the **maximum value of 3,556 rentals** represents **around five times the mean**, showing the presence of hours with exceptionally high demand.

In [12]:
# Descriptive statistics for most important weather variables

weather_stats = bike_df.select("Temperature_C","Humidity(%)","Wind speed (m/s)").summary()
weather_stats.show()

+-------+------------------+------------------+------------------+
|summary|     Temperature_C|       Humidity(%)|  Wind speed (m/s)|
+-------+------------------+------------------+------------------+
|  count|              8760|              8760|              8760|
|   mean|12.882922374429242|58.226255707762554|1.7249086757990955|
| stddev|11.944825230027963| 20.36241330156561|1.0362999934025572|
|    min|             -17.8|                 0|               0.0|
|    25%|               3.4|                42|               0.9|
|    50%|              13.7|                57|               1.5|
|    75%|              22.5|                74|               2.3|
|    max|              39.4|                98|               7.4|
+-------+------------------+------------------+------------------+



#### Descriptive statistics for weather features

##### Temperature (°C)

- **Mean:** **≈ 12.9 °C**
- **Standard deviation:** **≈ 11.9 °C** – indicates **high variability**, consistent with the large difference between the **minimum temperature** (**−17.8 °C**) and the **maximum** (**39.4 °C**).  
- The **range of values** (minimum and maximum), together with the **quartiles**, shows that the sample covers **all seasons of the year**.  
- The closeness between the **mean** and the **median** suggests an **approximately symmetric distribution**, since these values are similar.  
- In general, temperatures up to **22.5 °C** are the most common.

##### Humidity (%)

- **Mean:** **around 58%**
- **Standard deviation:** **≈ 20.4 p.p.** – shows **high variability**.  
- Humidity ranges from **very dry** to **very humid** conditions (**minimum 0% – maximum 98%**).  
- **Most observations** (between the **25th** and **75th percentiles**) are between **42%** and **74%**, covering **different situations**.

##### Wind speed (m/s)

- **Mean:** **≈ 1.72 m/s**
- **Standard deviation:** **≈ 1.04 m/s** – **relatively low variability**, indicating that even though the **maximum value** is **more than four times higher** than the **mean**, variability is still low.  
- In general, **light to moderate winds** prevail (**75% of the samples** did not exceed **2.3 m/s**, while the mean is **1.72 m/s**).

## View key features before running the queries

In [13]:
# Total number of records

bike_df.count()

8760

In [14]:
# View a few important features

bike_df.select("Date", "Hour", "Rented_Bike_Count", "Temperature_C").show(10)

+----------+----+-----------------+-------------+
|      Date|Hour|Rented_Bike_Count|Temperature_C|
+----------+----+-----------------+-------------+
|2017-12-07|   3|              102|          1.9|
|2017-12-08|  21|              336|         -3.3|
|2017-12-09|  11|              351|          0.0|
|2017-12-18|  17|              175|          0.4|
|2018-01-01|   9|              121|         -4.3|
|2018-01-02|   4|               31|         -2.7|
|2018-01-02|   5|               58|         -3.1|
|2018-01-03|   8|              670|         -7.0|
|2018-01-09|   3|               52|         -3.4|
|2018-01-26|  15|              140|        -11.4|
+----------+----+-----------------+-------------+
only showing top 10 rows



## Queries

### Query 1 - Verify if each season has a similar number of records
Before comparing the number of bikes rented in different situations, it is important to check whether the number of records in each season is reasonably balanced, so that the analysis is not biased.

In [15]:
from pyspark.sql.functions import count

(
    bike_df.groupBy("Seasons")                           # group by season
        .agg(count("*").alias("records_per_season"))     # count records and rename the column
        .orderBy("Seasons")                              # order alphabetically
        .show()                                          # display the result
)

+-------+------------------+
|Seasons|records_per_season|
+-------+------------------+
| Autumn|              2184|
| Spring|              2208|
| Summer|              2208|
| Winter|              2160|
+-------+------------------+



> This shows that the data are evenly distributed across all seasons, indicating that we can proceed with the remaining queries.

### Query 2 - Which season tends to have higher demand ?

Now that we know the proportion of records across seasons is reasonably balanced, we can compare the number of bike rentals in each season of the year.

In [16]:
from pyspark.sql.functions import avg, round

(
    bike_df.groupBy("Seasons")                                              # Group the dataframe by season
        .agg(round(avg("Rented_Bike_Count"), 2).alias("avg_rented_bikes"))  # Round the average and rename the column
        .orderBy("Seasons")                                                 # Order the result by seasons
        .show()                                                             # Display the result
)

+-------+----------------+
|Seasons|avg_rented_bikes|
+-------+----------------+
| Autumn|           819.6|
| Spring|          730.03|
| Summer|         1034.07|
| Winter|          225.54|
+-------+----------------+



> Summer records the highest average with 1,034 bikes, roughly four times more than Winter with 225. This confirms the expected pattern — warmer seasons drive higher demand.

### Query 3 - Average rented bikes by temperature band

Although this may seem obvious, we identified the expected pattern: in summer the number of rented bikes is the highest, while in winter it drops to roughly four times less. To understand this behaviour in more detail, instead of looking only at the seasons, we repeat the analysis using temperature. However, for analysis purposes, it is much more practical to create temperature bands rather than checking every individual temperature value.

In [17]:
from pyspark.sql.functions import when, avg, desc, count

categorized_df = bike_df.withColumn(
    "Temperature_Category",
    when(bike_df.Temperature_C <= 0, "≤ 0°C")
    .when((bike_df.Temperature_C > 0) & (bike_df.Temperature_C <= 5),  "0–5°C")
    .when((bike_df.Temperature_C > 5) & (bike_df.Temperature_C <= 10), "5–10°C")
    .when((bike_df.Temperature_C > 10) & (bike_df.Temperature_C <= 15), "10–15°C")
    .when((bike_df.Temperature_C > 15) & (bike_df.Temperature_C <= 20), "15–20°C")
    .when((bike_df.Temperature_C > 20) & (bike_df.Temperature_C <= 25), "20–25°C")
    .when((bike_df.Temperature_C > 25) & (bike_df.Temperature_C <= 30), "25–30°C")
    .otherwise("> 30°C")
)

category_analysis = (
    categorized_df
    .groupBy("Temperature_Category")
    .agg(
        count("*").alias("count_of_records"),
        avg("Rented_Bike_Count").alias("avg_rented_bikes")
    )
)

category_analysis_ordered = category_analysis.orderBy(desc("avg_rented_bikes"))
category_analysis_ordered.show()

+--------------------+----------------+------------------+
|Temperature_Category|count_of_records|  avg_rented_bikes|
+--------------------+----------------+------------------+
|             25–30°C|             990|1220.8343434343435|
|              > 30°C|             512|    1106.400390625|
|             20–25°C|            1397| 1048.530422333572|
|             15–20°C|            1222| 839.5147299509001|
|             10–15°C|            1042| 677.8176583493282|
|              5–10°C|            1079| 517.7933271547729|
|               0–5°C|            1064|330.29887218045116|
|               ≤ 0°C|            1454| 199.5213204951857|
+--------------------+----------------+------------------+



> Aiming to better understand this pattern, the records were grouped into several temperature ranges, with a range of 5 degrees per band, confirming that the higher the temperature, the more people rent bikes, with an exception above 30°C.

In [18]:
# Save the cleaned dataset to HDFS in Parquet format

bike_df.write.mode("overwrite").parquet("hdfs:///CA2_SBA25076/SeoulBikeData_clean")
print("Dataset saved to HDFS")

Dataset saved to HDFS


##  Final Conclusion

- After loading the dataset from HDFS and processing it with PySpark — leveraging its distributed computing capabilities to handle and transform the data efficiently — the analysis focused on understanding the factors that influence bike rental demand.

- The records were first aggregated by season, confirming that Summer drives the highest demand with an average of 1,034 bikes per hour, while Winter drops to 225 — roughly four times less.

- To go beyond seasonal analysis, temperature bands of 5-degree intervals were created using PySpark's `when()` function, allowing a more precise understanding of user behaviour.

- The results consistently confirmed that higher temperatures are associated with higher demand, peaking at the 25–30°C band with 1,220 bikes. However, above 30°C demand begins to fall, suggesting that extreme heat becomes uncomfortable.

- Although temperature proved to be a strong driver, it cannot be considered the only factor influencing demand, as other variables such as wind speed, humidity, and time of day were not explored in this section — aspects that will be addressed in Question 2.

# Question 2

In [19]:
from pyspark.sql import SparkSession
from pymongo import MongoClient

# Reusing the SparkSession created in Question 1
spark = SparkSession.builder.getOrCreate()

In [20]:
# Check if Hadoop process are running

!jps          

1265 DataNode
1555 SecondaryNameNode
2486 SparkSubmit
663 ResourceManager
1129 NameNode
2538 Jps
2061 SparkSubmit
799 NodeManager


### Loading Seoul Bike Sharing data from HDFS into MongoDB

In this step, I:

- Reuse the processed **Seoul Bike Sharing** dataset that I previously saved to **HDFS** in Parquet format using Spark.  
- Read this file from HDFS into a Spark DataFrame called `bike_df`.  

In [21]:
# Read the Seoul Bike dataset from HDFS into a Spark DataFrame


bike_df = spark.read.parquet("hdfs:///CA2_SBA25076/SeoulBikeData_clean")


bike_df.show(10, truncate=False)                             # Check a sample of rows to confirm the HDFS load


bike_df.printSchema()                                        # Print the type of each column

+----------+-----------------+----+-------------+-----------+----------------+----------------+----------+-----------------------+------------+-------------+-------+----------+---------------+
|Date      |Rented_Bike_Count|Hour|Temperature_C|Humidity(%)|Wind speed (m/s)|Visibility (10m)|DewPoint_C|Solar Radiation (MJ/m2)|Rainfall(mm)|Snowfall (cm)|Seasons|Holiday   |Functioning_Day|
+----------+-----------------+----+-------------+-----------+----------------+----------------+----------+-----------------------+------------+-------------+-------+----------+---------------+
|2017-12-07|102              |3   |1.9          |91         |1.4             |218             |0.5       |0.0                    |0.0         |0.9          |Winter |No Holiday|Yes            |
|2017-12-08|336              |21  |-3.3         |35         |0.5             |2000            |-16.6     |0.0                    |0.0         |0.0          |Winter |No Holiday|Yes            |
|2017-12-09|351              |11  |

> The preprocessing steps — column renaming, null and duplicate checks, and the conversion of the `Date` feature from string to date type — were already performed in Question 1. As verified there, the dataset contains no null values or duplicates. To avoid repetition, the cleaned dataset was saved to HDFS in Parquet format at the end of Question 1 and is loaded directly here. The PySpark data analysis was also carried out in Question 1. Following the same approach, and in order to keep the most relevant aspects of the assignment clear and well-structured, this section will focus on the MongoDB integration — covering data ingestion and query methods — rather than repeating steps that have been previously addressed.

### MongoDB Connection
- The connection is established on `127.0.0.1:27017`, which is MongoDB's default port, where 127.0.0.1 refers to the local machine itself.
- Store the data in a database named `ca2_sba25076` and a collection named `ca2_sba25076_bike_rentals`.

In [22]:
from pymongo import MongoClient

client = MongoClient("mongodb://127.0.0.1:27017")
client.admin.command("ping")  # test the connection

db = client["ca2_sba25076"]
collection = db["ca2_sba25076_bike_rentals"]

print("MongoDB connected")    # Check connection

MongoDB connected


In [23]:
collection.delete_many({})                                        # resets the collection before inserting
rows = bike_df.collect()                                          # converts the Spark DataFrame into a Python list for MongoDB insertion


documents = []                                                    # empty list to store the records

for row in rows:
    doc = {
        "date"              : str(row["Date"]),
        "hour"              : int(row["Hour"]),
        "rented_bike_count" : int(row["Rented_Bike_Count"]),
        "temperature_c"     : float(row["Temperature_C"]),
        "humidity"          : float(row["Humidity(%)"]),
        "wind_speed"        : float(row["Wind speed (m/s)"]),
        "visibility"        : int(row["Visibility (10m)"]),
        "dew_point_c"       : float(row["DewPoint_C"]),
        "solar_radiation"   : float(row["Solar Radiation (MJ/m2)"]),
        "rainfall_mm"       : float(row["Rainfall(mm)"]),
        "snowfall_cm"       : float(row["Snowfall (cm)"]),
        "seasons"           : str(row["Seasons"]),
        "holiday"           : str(row["Holiday"]),
        "functioning_day"   : str(row["Functioning_Day"]),
    }
    
    documents.append(doc)                                          # adds each document to the list
    
if documents:                                                      # Condition: only insert if the list is not empty
    collection.insert_many(documents)                              # inserts all documents into the collection at once
    
print(f"Loaded {len(documents)} rows into MongoDB collection")     # confirms how many documents were inserted

Loaded 8760 rows into MongoDB collection


## QUERIES

> Building on the findings from Question 1 — where temperature proved to be a strong driver of bike demand, with Summer recording the highest average rentals and the 25–30°C band standing out as the sweet spot — this section aims to go further. Using MongoDB as the query engine, the goal is to explore aspects that were not addressed previously: understanding whether the time of day and holiday status influence demand by analysing the top 15 hours with the highest average rentals, identifying the peak hours specifically during Summer, and finally investigating whether bikes are still rented under the most extreme conditions — sub-zero temperatures combined with winter season.

### Query 1 — Top 15 hours by average bikes rented, split by Holiday vs No Holiday

In [24]:
q1 = [
    {"$group": {
        "_id": {                                      # group by hour and holiday simultaneously                            
            "hour" : "$hour",                         # group by hour
            "holiday": "$holiday"                     # group by holidays status
        },
        "avg_bikes": {"$avg": "$rented_bike_count"}   # calculates the average bikes for each group
    }},
    {"$sort": {"avg_bikes": -1}},                     # order by avg
    {"$limit": 15}                                    # show 15 records
]


for doc in collection.aggregate(q1):                  # prints each result from the aggregation pipeline
    print(f"  Hour {doc['_id']['hour']:02d}h | {doc['_id']['holiday']:10s} → avg: {doc['avg_bikes']:.2f}")

  Hour 18h | No Holiday → avg: 1539.35
  Hour 19h | No Holiday → avg: 1219.69
  Hour 17h | No Holiday → avg: 1154.49
  Hour 20h | No Holiday → avg: 1087.39
  Hour 08h | No Holiday → avg: 1050.31
  Hour 21h | No Holiday → avg: 1049.77
  Hour 22h | No Holiday → avg: 939.71
  Hour 16h | No Holiday → avg: 938.04
  Hour 15h | No Holiday → avg: 833.41
  Hour 17h | Holiday    → avg: 830.50
  Hour 18h | Holiday    → avg: 800.67
  Hour 16h | Holiday    → avg: 787.61
  Hour 14h | No Holiday → avg: 760.11
  Hour 15h | Holiday    → avg: 747.78
  Hour 13h | No Holiday → avg: 734.83


> This query suggests that non-holiday days tend to have higher demand, peaking at 18h with an average of 1,539 bikes, indicating that, regardless of weather conditions, daily routines also play a relevant role in bike rental behaviour.

### Query 2 — Top 5 hours with highest demand during Summer

In [25]:
q2 = [
    {"$match": {"seasons" : "Summer"}},                # filter only Summer records
    {"$group": {
        "_id": "$hour",                                # group by hour
        "avg_bikes": {"$avg": "$rented_bike_count"}    # calculate average bikes
    }},
    {"$sort": {"avg_bikes": -1}},                      # order by highest average
    {"$limit": 5}                                      # limit 5 records
] 

for doc in collection.aggregate(q2):                   # prints each result from the aggregation pipeline
    print(f"  Hour {doc['_id']:02d}h → avg: {doc['avg_bikes']:.2f}")

  Hour 18h → avg: 2135.14
  Hour 19h → avg: 1889.25
  Hour 20h → avg: 1801.92
  Hour 21h → avg: 1754.07
  Hour 22h → avg: 1567.87


> Knowing that Summer presents the highest average rentals, this query confirms that peak demand occurs between 18h and 22h, reaching values of 2,135 bikes, reinforcing patterns already identified. 

> It is also worth noting that hours such as 12h do not appear in this top 5, suggesting that milder temperatures tend to be more attractive for bike rentals.

### Query 3 — Average and Maximum bikes rented during Winter with sub-zero temperatures

In [26]:
q3 = [
    {"$match": {
        "temperature_c": {"$lt": 0},  # below 0°C
        "seasons"      : "Winter"     # Winter season
    }},
    {"$group": {
        "_id"          : None,                           # no group, unique value
        "avg_bikes"    : {"$avg": "$rented_bike_count"}, # calculate average bikes
        "max_bikes"    : {"$max": "$rented_bike_count"}, # calculate max bikes
        "total_records": {"$sum": 1}                     # count total records
    }}
]

for doc in collection.aggregate(q3):  # prints each result from the aggregation pipeline
    print(f"  Records: {doc['total_records']} | avg bikes: {doc['avg_bikes']:.2f} | max bikes: {doc['max_bikes']}")

  Records: 1390 | avg bikes: 191.59 | max bikes: 937


> Despite the harsh conditions, 1,390 records were found with an average of 191 bikes and a maximum of 937, suggesting that a portion of users maintains this behaviour even under sub-zero temperatures.

## Final Conclusion

- Using MongoDB, it was possible to expand on the findings from the previous question, bringing in aspects that had not been previously addressed.

- The first query showed that not only temperature or season are relevant, but that daily routine also becomes a key factor — as evidenced by the peak hours at 08h and 18h–19h.

- The second query confirmed that even during Summer, the peak hours remain the same, reinforcing the findings from the previous query.

- In the third query, to gain a fuller understanding, the approach shifted slightly. Records from Winter with temperatures below 0°C were analysed. This showed that, despite the harsh conditions, rentals were still recorded — obviously in smaller numbers, but sufficient to confirm that this is a daily behaviour for many people who need to commute to work, study, and so on.

In [27]:
client.close()  # closes the MongoDB connection
print("MongoDB connection closed.")

MongoDB connection closed.


## References

- Databricks (2021) What is a Hadoop Distributed File System (HDFS)?. Available at: https://www.databricks.com/blog/what-is-hdfs (Accessed: 22 June 2026).

- Iqbal, M. (2026) Tutorial_8_realword_streaming. Lecture Notes. Dublin: CCT College Dublin.

- Iqbal, M. (2026) Tutorial Mongo. Lecture Notes. Dublin: CCT College Dublin.

- Apache.org. (2026). pyspark.sql.DataFrame.withColumnRenamed — PySpark 4.1.2 documentation. [online] Available at: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumnRenamed.html [Accessed 22 Jun. 2026].

- Bobbitt, Z. (2023). How to Count Null Values in PySpark (With Examples). [online] Statology. Available at: https://www.statology.org/pyspark-count-null-values/ [Accessed 22 Jun. 2026].

- Apache.org. (2026). pyspark.sql.functions.to_date — PySpark 4.1.2 documentation. [online] Available at: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_date.html [Accessed 22 Jun. 2026].

- Clemente, F. (2023). Single line of code data profiling with Spark. [online] Medium. Available at: https://medium.com/the-techlife/the-1-python-package-to-profile-your-spark-dataframes-89b09e37eeb9 [Accessed 22 Jun. 2026].

- Iqbal, M. (2026) MongoDB. Lecture Notes. Dublin: CCT College Dublin.

- Team, M.D. (2026). $match (aggregation stage). [online] Mongodb.com. Available at: https://www.mongodb.com/docs/manual/reference/operator/aggregation/match/#mongodb-pipeline-pipe.-match [Accessed 26 Jun. 2026].